In [1]:
import pandas as pd
import os

In [2]:
categorys = ["therapy","disease"]
df = pd.read_csv("./Profile_All.csv",usecols=["sample"]+categorys)
sample_list = df["sample"].tolist()

In [3]:
chains = []
for dirname,dirs,filenames in os.walk("./pep_data"):
    chains += dirs
chains

['IGL', 'IGK', 'TRA', 'TRB', 'TRG', 'TRD', 'IGH']

In [4]:
read_path = "pep_data"
for chain in chains:
    ###这里因为项目只有IGH 所以我们只考虑了IGH的情况
    topclone_dict = {"top10"+chain:{},"top20"+chain:{},"top50"+chain:{},"top100"+chain:{}}
    topCDR3_dict = {"top10"+chain:{},"top20"+chain:{},"top50"+chain:{},"top100"+chain:{}}
    for sample in sample_list:
        df_pep = pd.read_csv(read_path+"/"+chain+"/"+sample+"__"+chain+".csv")
        df_pep = df_pep[["CDR3(pep)","copy"]]
        df_pep_CDR3copy = df_pep.groupby(by="CDR3(pep)").sum().reset_index()
        df_copy = df_pep_CDR3copy[~df_pep_CDR3copy["CDR3(pep)"].str.contains("\*|_")].sort_values("copy",ascending=False)
        topCDR3_dict["top10"+chain][sample] = df_copy["CDR3(pep)"].iloc[:10].tolist()
        topCDR3_dict["top20"+chain][sample] = df_copy["CDR3(pep)"].iloc[:20].tolist()
        topCDR3_dict["top50"+chain][sample] = df_copy["CDR3(pep)"].iloc[:50].tolist()
        topCDR3_dict["top100"+chain][sample] = df_copy["CDR3(pep)"].iloc[:100].tolist()

        top10_proportion = df_copy["copy"].iloc[:10].sum()/df_copy["copy"].sum()
        top20_proportion = df_copy["copy"].iloc[:20].sum()/df_copy["copy"].sum()
        top50_proportion = df_copy["copy"].iloc[:50].sum()/df_copy["copy"].sum()
        top100_proportion = df_copy["copy"].iloc[:100].sum()/df_copy["copy"].sum()
        topclone_dict["top10"+chain][sample] = top10_proportion
        topclone_dict["top20"+chain][sample] = top20_proportion
        topclone_dict["top50"+chain][sample] = top50_proportion
        topclone_dict["top100"+chain][sample] = top100_proportion
    topclone_df = pd.DataFrame(topclone_dict)
    topclone_df.index.name = "sample"
    topclone_df = topclone_df.reset_index()
    df = pd.merge(df,topclone_df,on="sample")

/tmp/ipykernel_1391478/3205054900.py:7: DtypeWarning: Columns (6) have mixed types. Specify dtype option on import or set low_memory=False.
  df_pep = pd.read_csv(read_path+"/"+chain+"/"+sample+"__"+chain+".csv")
/tmp/ipykernel_1391478/3205054900.py:7: DtypeWarning: Columns (6) have mixed types. Specify dtype option on import or set low_memory=False.
  df_pep = pd.read_csv(read_path+"/"+chain+"/"+sample+"__"+chain+".csv")
/tmp/ipykernel_1391478/3205054900.py:7: DtypeWarning: Columns (6,17,18,19,20) have mixed types. Specify dtype option on import or set low_memory=False.
  df_pep = pd.read_csv(read_path+"/"+chain+"/"+sample+"__"+chain+".csv")
/tmp/ipykernel_1391478/3205054900.py:7: DtypeWarning: Columns (6) have mixed types. Specify dtype option on import or set low_memory=False.
  df_pep = pd.read_csv(read_path+"/"+chain+"/"+sample+"__"+chain+".csv")
/tmp/ipykernel_1391478/3205054900.py:7: DtypeWarning: Columns (6) have mixed types. Specify dtype option on import or set low_memory=Fal

In [5]:
df.to_csv("./topclone.csv",index=False)